# Conditional Imitation Learning - Proyecto Final MR4010 - Equipo 18

**MR4010 - Navegación Autónoma | MNA | Tecnológico de Monterrey**

**Equipo 18:**
- Alexis Alduncin Barragán — A01017478
- David Rodrigo Alvarado Domínguez
- Abraham Avila Garcia
- Jorge Luis Ancheyta Segovia

Este notebook entrena la red CNN de **Conditional Imitation Learning (CIL)** siguiendo la
arquitectura *ramificada* (branched) de Codevilla et al. (2018). El modelo aprende a conducir
por imitación a partir de los frames y comandos de navegación capturados con el controlador de
recolección de datos en Webots. La cámara entrega imágenes de 200×88×3 y, junto con un comando
de alto nivel (seguir carril, girar izq./der., seguir recto), la red predice el ángulo de
dirección (*steering*). El comando actúa como un **switch** que selecciona cuál de las 4 ramas
especialistas produce la predicción. El notebook corre igual en local (Mac, prueba de hoy) y en
Google Colab con GPU (dataset consolidado del equipo, viernes).

**Cita:** Codevilla, F., Müller, M., López, A., Koltun, V., & Dosovitskiy, A. (2018).
*End-to-end Driving via Conditional Imitation Learning.* IEEE International Conference on
Robotics and Automation (ICRA).


In [ ]:
import os, sys, json, time, importlib
from pathlib import Path

# Detectamos el entorno de ejecución: Colab (viernes, GPU) vs local (Mac, prueba de hoy).
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print(f"Running in: {'Colab' if IN_COLAB else 'local'}")

# ── (1) Imports de paquetes PREINSTALADOS primero ────────────────────────────
# Los importamos ANTES del pip install por dos razones:
#   (a) capturar sus versiones EXACTAS para fijarlas (pin) en el install, y
#   (b) si algo truena, sabemos que el culpable es el pip install y no estos.
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
import keras                      # Keras 3 standalone (expone keras.saving para capas custom)
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
print(f"Pre-install → TensorFlow {tf.__version__} | Keras {keras.__version__} | numpy {np.__version__}")

# ── (2) Instalar albumentations SOLO en Colab, SIN romper el stack ───────────
# El env local 'webots' ya tiene albumentations; en Colab hay que instalarlo.
# CLAVE: fijamos tensorflow / keras / numpy a las versiones YA preinstaladas para
# que el resolver de pip NO pueda moverlas. Un `pip install albumentations` "pelón"
# puede bumpear numpy y romper el binario compilado de TensorFlow; pip entonces
# DESINSTALA tensorflow → `ModuleNotFoundError: No module named 'tensorflow'` en el
# siguiente import. Al fijarlas, si hubiera un conflicto real pip falla RUIDOSAMENTE
# aquí en vez de dejar el entorno roto en silencio.
if IN_COLAB:
    pins = (f"albumentations==2.0.8 "
            f"tensorflow=={tf.__version__} "
            f"keras=={keras.__version__} "
            f"numpy=={np.__version__}")
    !pip install -q {pins}

# ── (3) Sanity-check DURO: si algún import quedó roto, abortamos con mensaje claro ──
# Convierte un entorno a medias en un error temprano y legible, en vez de un fallo
# críptico tres celdas más abajo. Incluye albumentations (recién instalado en Colab).
REQUIRED = ['tensorflow', 'keras', 'numpy', 'pandas', 'cv2', 'matplotlib',
            'albumentations', 'sklearn']
missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError as e:
        missing.append((mod, str(e)))
if missing:
    print("❌ MÓDULOS FALTANTES — la configuración de la Celda 1 falló:")
    for mod, err in missing:
        print(f"   {mod}: {err}")
    raise RuntimeError("Celda 1 incompleta. NO ejecutes las celdas siguientes.")
print("✓ Todos los módulos requeridos son importables")

# ── (4) Bind del namespace que usan las celdas siguientes (post-check) ───────
import albumentations as A

print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {keras.__version__}")
print(f"Albumentations: {A.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

# ─────────────────────────────────────────────────────────────────────────────
# Resolución de la ruta del dataset (funciona en local y en Colab sin editar celdas)
# ─────────────────────────────────────────────────────────────────────────────
if IN_COLAB:
    # ─────────────────────────────────────────────────────────
    # CONFIG: set to 'github' for production (Friday consolidated dataset)
    #         set to 'drive' for today's smoke test on alexis's data only
    DATASET_SOURCE = 'drive'  # ← Alexis flips this to 'github' on Friday
    # ─────────────────────────────────────────────────────────

    if DATASET_SOURCE == 'github':
        # Friday: dataset lives in its own repo per project spec
        # !git clone https://github.com/AlexisAlduncintec/navegacion_autonoma_dataset.git /content/dataset
        # !cd /content/dataset && unzip -q '*.zip'  # if chunks are zipped
        # DATASET_ROOT = '/content/dataset'
        raise NotImplementedError("Friday: uncomment the clone+unzip lines above")
    elif DATASET_SOURCE == 'drive':
        from google.colab import drive
        drive.mount('/content/drive')
        # Today: alexis's zip lives in 00_dataset_chunks/alexis/alexis_20260625.zip
        # Need to unzip into a working dir first
        import zipfile, os
        ZIP_PATH = '/content/drive/MyDrive/MR4010_Proyecto_Final_Equipo18/00_dataset_chunks/alexis/alexis_20260625.zip'
        WORK_DIR = '/content/dataset_unzipped'
        if not os.path.exists(WORK_DIR):
            os.makedirs(WORK_DIR)
            print(f"Unzipping {ZIP_PATH} → {WORK_DIR}...")
            with zipfile.ZipFile(ZIP_PATH) as zf:
                zf.extractall(WORK_DIR)
            print(f"Unzipped {len(os.listdir(os.path.join(WORK_DIR, 'IMG')))} session folders")
        DATASET_ROOT = WORK_DIR
    else:
        raise ValueError(f"Unknown DATASET_SOURCE: {DATASET_SOURCE}")
else:
    # Prueba local en la Mac de Alexis.
    DATASET_ROOT = '/Users/alexisalduncin/ITESM/navegacion_autonoma/Proyecto_Final/dataset'

print(f"Dataset root: {DATASET_ROOT}")
CSV_PATH = os.path.join(DATASET_ROOT, 'driving_log.csv')
print(f"CSV exists: {os.path.exists(CSV_PATH)}")


## 2. Carga e inspección del dataset

Cargamos el `driving_log.csv` generado por el controlador de recolección. Cada fila es un frame
con su imagen, el ángulo de dirección etiquetado, el comando de navegación de alto nivel, y
metadatos de sesión. Revisamos la distribución de comandos (clave para CIL: cada comando entrena
una rama distinta), las estadísticas de steering y velocidad, y hacemos una inspección visual.

**Higiene de datos:** filtramos comandos fuera de {2,3,4,5} y filas con steering/command NaN.
Hoy el dataset de Alexis está limpio, pero el viernes consolidamos datos de 4 personas, así que
esta guarda evita que un frame corrupto indexe una rama inexistente.


In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Total frames (raw): {len(df)}")

# Higiene de datos: comandos válidos y sin NaN en las columnas que sí usamos.
CMD_NAMES = {2: 'FOLLOW_LANE', 3: 'TURN_LEFT', 4: 'TURN_RIGHT', 5: 'GO_STRAIGHT'}
df = df.dropna(subset=['steering', 'command'])
df = df[df['command'].isin(list(CMD_NAMES.keys()))].reset_index(drop=True)
df['command'] = df['command'].astype(int)
print(f"Total frames (clean): {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst rows:")
print(df.head())

# Mapeo de comando a nombre legible (Codevilla §V).
df['command_name'] = df['command'].map(CMD_NAMES)

# Distribución de comandos.
print(f"\nCommand distribution:")
for cmd, count in df['command'].value_counts().sort_index().items():
    print(f"  {CMD_NAMES[cmd]} ({cmd}): {count} ({100*count/len(df):.1f}%)")

print(f"\nSteering stats: min={df['steering'].min():.3f}, max={df['steering'].max():.3f}, "
      f"mean={df['steering'].mean():.3f}, std={df['steering'].std():.3f}")
print(f"Speed stats: min={df['speed'].min():.2f}, max={df['speed'].max():.2f}")
print(f"Sessions: {df['session_id'].nunique()}")

# Visualizamos las distribuciones (estas gráficas van al reporte).
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df['command'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color=['white', 'cyan', 'magenta', 'yellow'], edgecolor='black')
axes[0].set_title('Comando de navegación')
axes[0].set_xticklabels([CMD_NAMES[c] for c in sorted(df['command'].unique())], rotation=30)
axes[1].hist(df['steering'], bins=50, edgecolor='black')
axes[1].set_title('Distribución de steering angle')
axes[1].set_xlabel('rad')
axes[2].hist(df['speed'], bins=30, edgecolor='black')
axes[2].set_title('Distribución de velocidad')
axes[2].set_xlabel('km/h')
plt.tight_layout()
plt.savefig('dataset_distribution.png', dpi=120)
plt.show()

# Inspección visual: una imagen por comando (tomamos la del medio de cada grupo).
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for i, cmd in enumerate(sorted(df['command'].unique())):
    grp = df[df['command'] == cmd]
    sample = grp.iloc[len(grp) // 2]
    img_path = os.path.join(DATASET_ROOT, sample['image_path'])
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)
    axes[i].set_title(f"{CMD_NAMES[cmd]}\nsteer={sample['steering']:.3f}")
    axes[i].axis('off')
plt.tight_layout()
plt.savefig('dataset_samples.png', dpi=120)
plt.show()


## 3. División train/val (80/20 estratificado)

Dividimos 80% entrenamiento / 20% validación **estratificando por comando**. Esto garantiza que
la proporción de cada comando (FOLLOW_LANE, TURN_LEFT, TURN_RIGHT, GO_STRAIGHT) sea la misma en
ambos conjuntos. Es importante porque las clases están desbalanceadas (GO_STRAIGHT domina) y cada
comando alimenta una rama distinta del modelo: si una rama recibiera pocos ejemplos en validación,
su métrica sería poco confiable.


In [ ]:
# Estratificamos por comando para mantener la distribución balanceada en ambos splits.
train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['command'])
print(f"Train: {len(train_df)} ({100*len(train_df)/len(df):.1f}%)")
print(f"Val:   {len(val_df)} ({100*len(val_df)/len(df):.1f}%)")

print(f"\nTrain command distribution:")
for cmd, count in train_df['command'].value_counts().sort_index().items():
    print(f"  {CMD_NAMES[cmd]}: {count} ({100*count/len(train_df):.1f}%)")

print(f"\nVal command distribution:")
for cmd, count in val_df['command'].value_counts().sort_index().items():
    print(f"  {CMD_NAMES[cmd]}: {count} ({100*count/len(val_df):.1f}%)")


## 4. Pipeline de datos + data augmentation

Construimos un `tf.data.Dataset` que lee las imágenes bajo demanda y aplica augmentación.

**Decisión de diseño — augmentación FOTOMÉTRICA únicamente (Codevilla §IV.D):** usamos
`albumentations` (NO `imgaug`, que está muerto e incompatible con NumPy 2.x). Aplicamos solo
transformaciones de color/ruido (brillo, contraste, tono, blur, ruido, dropout). **NO** usamos
transformaciones geométricas (flip, rotación, traslación): cambiarían la geometría aparente de la
carretera e **invalidarían la etiqueta de steering**. El preprocesamiento es solo normalizar a
[0, 1]; NO convertimos a YUV (eso es la receta de Bojarski/PilotNet, no la de Codevilla CIL).


In [ ]:
IMG_H, IMG_W = 88, 200  # Codevilla exact (la cámara de Webots entrega 200×88)

# Pipeline de augmentación de Codevilla §IV.D — SOLO FOTOMÉTRICA.
augment_train = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=15, p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.GaussNoise(p=0.2),
    A.CoarseDropout(num_holes_range=(1, 3), hole_height_range=(4, 8), hole_width_range=(4, 8), p=0.3),
])
# Validación: sin augmentación, evaluación limpia.
augment_val = A.Compose([])

# Preprocesamiento Codevilla: solo normalizar a [0, 1]. SIN conversión YUV.
def preprocess(img):
    return img.astype(np.float32) / 255.0

def make_dataset(df_split, augment, batch_size=120, shuffle=True):
    """Construye un tf.data.Dataset a partir de un split (DataFrame)."""

    def gen():
        idx = list(range(len(df_split)))
        if shuffle:
            np.random.shuffle(idx)
        for i in idx:
            row = df_split.iloc[i]
            img_path = os.path.join(DATASET_ROOT, row['image_path'])
            img = cv2.imread(img_path)
            if img is None:
                # Imagen faltante o corrupta: la saltamos en lugar de romper el epoch.
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            # Guarda defensiva: forzamos el tamaño exacto que espera la red.
            if img.shape[0] != IMG_H or img.shape[1] != IMG_W:
                img = cv2.resize(img, (IMG_W, IMG_H))
            img = augment(image=img)['image']
            img = preprocess(img)

            # Índice de comando (2,3,4,5) -> (0,1,2,3) para seleccionar la rama.
            cmd_idx = int(row['command']) - 2
            steering = np.float32(row['steering'])

            yield img, cmd_idx, steering

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=(IMG_H, IMG_W, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.int32),
            tf.TensorSpec(shape=(), dtype=tf.float32),
        )
    )
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# Smoke test del pipeline.
print("Smoke testing pipeline...")
test_ds = make_dataset(train_df.head(240), augment_train, batch_size=4)
for img_batch, cmd_batch, steer_batch in test_ds.take(1):
    print(f"img batch shape: {img_batch.shape}, dtype: {img_batch.dtype}")
    print(f"cmd batch shape: {cmd_batch.shape}, values: {cmd_batch.numpy()}")
    print(f"steer batch shape: {steer_batch.shape}, values: {steer_batch.numpy()}")
print("Pipeline OK")


## 5. Arquitectura CIL ramificada (Codevilla 2018, Figura 3b)

Implementamos la variante **ramificada** (branched), que el paper reporta superior a la variante
*command-input* (88% vs 78% de éxito). Tres piezas:

1. **Módulo de imagen (percepción compartida):** 8 capas convolucionales + 2 capas densas (512),
   con BatchNorm + ReLU + Dropout. Extrae un vector de características de 512-d.
2. **Módulo de mediciones — OMITIDO:** no pasamos la velocidad como entrada porque conducimos a
   crucero constante (~25 km/h); no aporta señal.
3. **4 ramas especialistas:** una por comando. Cada rama tiene 2 densas (256) + una salida de
   steering con `tanh` (rango [-1, 1]). El comando actúa como **switch**: con `tf.gather_nd`
   seleccionamos, por cada muestra del batch, la predicción de la rama correspondiente.

**Nota de implementación:** el switch se implementa como una capa custom `CommandSwitch`
registrada con `@keras.saving.register_keras_serializable()`. Así el modelo se vuelve a cargar
limpio desde `.h5` en Keras 3 (un `Lambda` con función custom no se puede deserializar del formato
H5 legacy). La salida seleccionada y la etiqueta son ambas de rango 1 `(batch,)`; mantenerlas
alineadas evita un *broadcast* silencioso `(batch,1)` vs `(batch,)` que corrompería el MSE.


In [ ]:
@keras.saving.register_keras_serializable()
class CommandSwitch(layers.Layer):
    """Switch de Codevilla: por cada muestra del batch selecciona la salida de la rama
    indicada por el comando. Entradas: (outputs (batch,4), command (batch,)). Salida: (batch,).
    Implementado como capa registrada para que el modelo recargue limpio desde .h5 (Keras 3)."""
    def call(self, inputs):
        outs, cmd = inputs
        batch_size = tf.shape(outs)[0]
        indices = tf.stack([tf.range(batch_size), cmd], axis=1)
        return tf.gather_nd(outs, indices)  # (batch,)

    def compute_output_shape(self, input_shapes):
        return (input_shapes[0][0],)


def build_image_module(input_shape=(IMG_H, IMG_W, 3)):
    """Módulo de imagen de Codevilla §IV.B: 8 conv + 2 FC."""
    inp = layers.Input(shape=input_shape, name='image_input')
    x = inp

    # Capas convolucionales (Codevilla: kernel 5 en la primera capa, 3 después;
    # strides 2 en capas 1/3/5; canales 32→64→128→256).
    x = layers.Conv2D(32, 5, strides=2, padding='same', activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(32, 3, strides=1, padding='same', activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(64, 3, strides=2, padding='same', activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(64, 3, strides=1, padding='same', activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(128, 3, strides=2, padding='same', activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(128, 3, strides=1, padding='same', activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(256, 3, strides=1, padding='same', activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(256, 3, strides=1, padding='same', activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(512, activation=None)(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation=None)(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.5)(x)

    return keras.Model(inp, x, name='image_module')

def build_branch(features_dim=512, name='branch'):
    """Una rama especialista: 2 FC + salida de steering."""
    inp = layers.Input(shape=(features_dim,))
    x = layers.Dense(256, activation=None)(inp)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation=None)(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation='tanh', name=f'{name}_steering')(x)  # tanh → [-1,1]
    return keras.Model(inp, out, name=name)

def build_cil_model():
    """CIL ramificado de Codevilla: módulo de imagen compartido + 4 ramas especialistas.
    El índice de comando (0-3) selecciona qué predicción de rama se usa."""
    image_inp = layers.Input(shape=(IMG_H, IMG_W, 3), name='image')
    command_inp = layers.Input(shape=(), dtype=tf.int32, name='command')

    # Percepción compartida.
    image_module = build_image_module()
    features = image_module(image_inp)

    # 4 ramas especialistas (Codevilla Figura 3b).
    branch_follow   = build_branch(name='branch_follow')
    branch_left     = build_branch(name='branch_left')
    branch_right    = build_branch(name='branch_right')
    branch_straight = build_branch(name='branch_straight')

    out_follow   = branch_follow(features)    # (batch, 1)
    out_left     = branch_left(features)
    out_right    = branch_right(features)
    out_straight = branch_straight(features)

    # Apilamos en (batch, 4) y seleccionamos por índice de comando.
    all_outputs = layers.Concatenate(axis=-1)(
        [out_follow, out_left, out_right, out_straight])  # (batch, 4)

    # Switch por comando: rango 1 (batch,) para alinear con la etiqueta (batch,) y evitar broadcast.
    selected = CommandSwitch(name='command_switch')([all_outputs, command_inp])

    model = keras.Model(inputs=[image_inp, command_inp], outputs=selected, name='cil_branched')
    return model

# Construimos y resumimos.
model = build_cil_model()
model.summary()

# Compilamos: pérdida MSE sobre steering (Codevilla ec. 6, variante de un solo objetivo).
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=2e-4),
    loss='mse',
    metrics=['mae'],
)

# Conteo de parámetros y tamaño en disco estimado.
n_params = model.count_params()
print(f"\nTotal params: {n_params:,}")
model_size_mb = model.count_params() * 4 / (1024**2)  # float32 = 4 bytes
print(f"Estimated model size: ~{model_size_mb:.1f} MB")


## 6. Entrenamiento

Entrenamos con Adam (`lr=2e-4`), batch 120 (valores exactos de Codevilla), hasta 25 epochs.
Usamos `EarlyStopping` sobre `val_loss` (restaura los mejores pesos), `ModelCheckpoint` que guarda
el mejor modelo como `cil_model.h5`, y `ReduceLROnPlateau`. Antes de `fit` hacemos un **pre-flight**:
pasamos un batch por el modelo para verificar el alineamiento de formas y fallar al instante en vez
de a los 5 minutos del primer epoch.


In [ ]:
BATCH = 120
EPOCHS = 25

train_ds = make_dataset(train_df, augment_train, batch_size=BATCH, shuffle=True)
val_ds   = make_dataset(val_df,   augment_val,   batch_size=BATCH, shuffle=False)

# El dataset entrega (img, cmd, steer) pero Keras .fit espera ((inputs), target). Lo reempaquetamos.
def reshape_for_fit(img, cmd, steer):
    return (img, cmd), steer

train_ds = train_ds.map(reshape_for_fit)
val_ds   = val_ds.map(reshape_for_fit)

# Pre-flight: pull ONE batch through the model to surface shape errors
# immediately instead of failing 5 minutes into the first epoch.
for sample_inputs, sample_target in train_ds.take(1):
    sample_pred = model(sample_inputs, training=False)
    print(f"Pre-flight: input image {sample_inputs[0].shape}, input cmd {sample_inputs[1].shape}, "
          f"target {sample_target.shape}, pred {sample_pred.shape}")
    assert sample_pred.shape == sample_target.shape, (
        f"SHAPE MISMATCH: pred {sample_pred.shape} vs target {sample_target.shape}")
    print("Pre-flight OK\n")

callbacks = [
    keras.callbacks.ModelCheckpoint('cil_model.h5', save_best_only=True, monitor='val_loss', verbose=1),
    keras.callbacks.EarlyStopping(patience=5, monitor='val_loss', restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, monitor='val_loss', min_lr=1e-6, verbose=1),
]

print(f"Training: {EPOCHS} epochs max, batch {BATCH}, train_n={len(train_df)}, val_n={len(val_df)}")
print(f"Estimated steps/epoch: ~{len(train_df)//BATCH}")

t0 = time.time()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)
print(f"\nTraining time: {(time.time()-t0)/60:.1f} min")

# Graficamos las curvas de pérdida y MAE.
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(history.history['loss'], label='train')
ax[0].plot(history.history['val_loss'], label='val')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('MSE loss')
ax[0].set_title('Loss')
ax[0].legend()
ax[0].grid(alpha=0.3)
ax[1].plot(history.history['mae'], label='train')
ax[1].plot(history.history['val_mae'], label='val')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('MAE (rad)')
ax[1].set_title('Mean Absolute Error')
ax[1].legend()
ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('cil_training_loss.png', dpi=120)
plt.show()

# Métricas finales.
print(f"\nFinal training loss: {history.history['loss'][-1]:.6f}")
print(f"Final validation loss: {history.history['val_loss'][-1]:.6f}")
print(f"Final training MAE: {history.history['mae'][-1]:.4f} rad")
print(f"Final validation MAE: {history.history['val_mae'][-1]:.4f} rad")
print(f"\nBest model saved to: cil_model.h5")
print(f"Loss curves saved to: cil_training_loss.png")


## 7. Sanity check de predicciones

Recargamos el mejor modelo guardado y comparamos la predicción de steering contra el valor real
en 8 muestras aleatorias de validación. Verde = error < 0.05; naranja = error mayor. Es un chequeo
rápido de cordura, no una evaluación formal.

**Nota Keras 3:** `load_model` recarga limpio porque el switch es una capa `CommandSwitch`
registrada con `@keras.saving.register_keras_serializable()` (definida en la celda 5, que debe
haberse ejecutado antes).


In [ ]:
# Recargamos los mejores pesos. CommandSwitch está registrada → recarga sin custom_objects.
model = keras.models.load_model('cil_model.h5', compile=False)

sample_df = val_df.sample(8, random_state=0)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (_, row) in zip(axes.flatten(), sample_df.iterrows()):
    img = cv2.imread(os.path.join(DATASET_ROOT, row['image_path']))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if img.shape[0] != IMG_H or img.shape[1] != IMG_W:
        img = cv2.resize(img, (IMG_W, IMG_H))
    img_pre = preprocess(img)[np.newaxis, ...]
    cmd = np.array([int(row['command']) - 2], dtype=np.int32)
    pred = float(np.reshape(model.predict([img_pre, cmd], verbose=0), -1)[0])
    ax.imshow(img)
    ax.set_title(f"{CMD_NAMES[row['command']]}\ntrue={row['steering']:+.3f}  pred={pred:+.3f}",
                 color='green' if abs(pred - row['steering']) < 0.05 else 'orange')
    ax.axis('off')
plt.tight_layout()
plt.savefig('cil_sanity_check.png', dpi=120)
plt.show()
